# Day 3 — Airport Operations AI Agent

This notebook builds the agentic decision-making layer for the Airport Operations AI Copilot.

The agent combines airport policy knowledge from the Day 1 RAG pipeline with operational tools from Day 2.

The goal is to allow the LLM to determine when it needs policy information, operational metrics, or an operational tool.

The agent should use retrieved policy information and tool results instead of inventing facts.

In [1]:
# Certificate issue resolve
import os

os.environ['REQUESTS_CA_BUNDLE'] = '/etc/ssl/certs/ca-certificates.crt'
os.environ['SSL_CERT_FILE'] = '/etc/ssl/certs/ca-certificates.crt'

import sys
from pathlib import Path

import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
from dotenv import load_dotenv
from google import genai
from google.genai import types

# Add the project root so notebook can import project files
sys.path.append("..")

print("Environment ready")

/home/nineleaps/Documents/da_python/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Environment ready


## 1. Load the Existing Agent Components

Day 3 builds on the components already implemented during Days 1 and 2.

The policy documents and vector search provide grounded policy knowledge, while the operational tools provide current airport metrics and controlled actions.

These components will now be connected to Gemini so the model can decide which information or tool is required for a user request.

In [2]:
# Load the Gemini API key
load_dotenv()

api_key = os.getenv("GEMINI_API_KEY")

# Create the Gemini client
client = genai.Client(api_key=api_key)

# Load the embedding model used by the Day 1 RAG pipeline
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Import the operational tools created on Day 2
from src.tools import (
    get_airport_metrics,
    calculate_driver_incentive,
    trigger_surge_override
)

print("API key available:", bool(api_key))
print("Embedding model loaded")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3081.41it/s]


API key available: True
Embedding model loaded


## 2. Load the Policy Knowledge Base

The Policy & Compliance Agent needs access to the same policy knowledge base created on Day 1.

The existing documents are loaded, split into chunks, embedded using the same Sentence Transformer model, and indexed with FAISS.

This allows the policy agent to retrieve relevant airport rules when evaluating an operational issue.

In [3]:
# Load the policy documents created on Day 1
POLICY_DIR = Path("../data/airport_policies")

policy_files = list(POLICY_DIR.glob("*.md"))

documents = []
for file_path in policy_files:
    documents.append({
        "source": file_path.name,
        "text": file_path.read_text(encoding="utf-8")
    })

print("Policy documents:", len(documents))

Policy documents: 9


In [4]:
# Split policy text into overlapping chunks for retrieval
def chunk_text(text, chunk_size=500, overlap=100):
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap

    return chunks


chunks = []

for doc in documents:
    for chunk in chunk_text(doc["text"]):
        chunks.append({
            "text": chunk,
            "source": doc["source"]
        })

print("Policy chunks:", len(chunks))

Policy chunks: 19


In [5]:
# Create embeddings for all policy chunks
texts = [chunk["text"] for chunk in chunks]

policy_embeddings = embedding_model.encode(
    texts,
    convert_to_numpy=True
)

# Normalize vectors so inner product represents cosine similarity
faiss.normalize_L2(policy_embeddings)

# Build the FAISS similarity index
policy_index = faiss.IndexFlatIP(policy_embeddings.shape[1])
policy_index.add(policy_embeddings)

print("Indexed policy vectors:", policy_index.ntotal)

Indexed policy vectors: 19


In [6]:
# Retrieve the most relevant policy chunks for a question
def search_policy(query, top_k=3):
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    )

    faiss.normalize_L2(query_embedding)

    scores, indices = policy_index.search(
        query_embedding,
        top_k
    )

    results = []

    for score, idx in zip(scores[0], indices[0]):
        result = chunks[idx].copy()
        result["score"] = float(score)
        results.append(result)

    return results


# Test policy retrieval
policy_results = search_policy(
    "What is the maximum surge allowed at SFO?"
)

for result in policy_results:
    print(result["source"], result["score"])

sfo_pricing.md 0.8108595609664917
lax_pricing.md 0.621222734451294
jfk_pricing.md 0.6035785675048828


## 3. Define the Specialized Agents

The system uses three specialized agents, each with a focused responsibility.

The Operations Investigator works with live operational data and identifies anomalies.

The Policy & Compliance Agent retrieves relevant policy information and checks restrictions.

The Resolution Agent uses the investigation and policy findings to produce an operational recommendation.

In [7]:
# Define the three specialized agent roles
investigator_instruction = """
You are the Operations Investigator Agent.

Analyze airport telemetry provided through operational tools.
Identify anomalies and determine their severity.
Use evidence such as completion rate, ETA, driver cancellations, and queue size.
Do not invent operational metrics.
"""

policy_instruction = """
You are the Policy & Compliance Agent.

Use the airport policy knowledge base to retrieve relevant policies.
Identify applicable limits, restrictions, and approval requirements.
Do not invent policy rules.
"""

resolution_instruction = """
You are the Resolution Agent.

Use the investigation findings and applicable policy.
Generate practical possible interventions and a recommended resolution.
Do not execute sensitive actions.
Clearly explain the reasoning using the available evidence.
"""

print("Three agent roles defined")

Three agent roles defined


## 4. Build Agent Handoff Logic

The Orchestrator coordinates the specialized agents rather than performing every task itself.

A typical operational investigation follows:
Orchestrator → Operations Investigator → Policy & Compliance → Resolution.

The handoff allows each agent to work only on the part of the problem relevant to its responsibility.

In [8]:
# Store the specialized agents in a simple agent map
agents = {
    "investigator": investigator_instruction,
    "policy": policy_instruction,
    "resolution": resolution_instruction
}

# Define the orchestrator's responsibility
orchestrator_instruction = """
You are the Orchestrator Agent for an Airport Operations AI Copilot.

Coordinate the specialized agents:
1. Operations Investigator
2. Policy & Compliance Agent
3. Resolution Agent

Determine what information is required and which agent should handle it.
Ensure operational claims come from tools and policy claims come from retrieval.
"""

## 5. Operations Investigator Agent

The Operations Investigator is responsible for examining airport telemetry.

It first obtains the latest airport metrics using the operational tool.
It then identifies whether the airport has an operational anomaly and determines its severity.

The investigator must base its findings only on the returned operational data.

In [9]:
# Investigate the latest operational state of an airport
def investigate_airport(airport_code):
    metrics = get_airport_metrics(airport_code)

    # Stop if the operational tool returned an error
    if metrics["status"] == "error":
        return metrics

    # Check whether completion rate is below the project threshold
    if metrics["completion_rate"] < 0.85:
        severity = "high"
        issue = "Low completion rate"
    elif metrics["completion_rate"] < 0.90:
        severity = "medium"
        issue = "Below-normal completion rate"
    else:
        severity = "low"
        issue = "No major completion-rate anomaly"

    return {
        "status": "success",
        "airport_code": airport_code,
        "metrics": metrics,
        "severity": severity,
        "issue": issue
    }


# Test the investigator
investigation = investigate_airport("SFO")

print(investigation)

{'status': 'success', 'airport_code': 'SFO', 'metrics': {'status': 'success', 'airport_code': 'SFO', 'completion_rate': 0.86, 'average_eta': 12, 'active_drivers': 500, 'driver_cancellation_rate': 0.1, 'queue_size': 100, 'surge_multiplier': 1.1}, 'severity': 'medium', 'issue': 'Below-normal completion rate'}


## 6. Policy & Compliance Agent

The Policy & Compliance Agent retrieves policies relevant to the operational issue.

For example, if the proposed resolution involves changing surge, the agent searches the airport pricing policy.

The retrieved policy is then supplied to the Resolution Agent so that recommendations remain grounded in documented rules.

In [10]:
# Retrieve policies relevant to an operational issue
def policy_agent(query):
    results = search_policy(query, top_k=3)

    # Combine the retrieved policy chunks
    context = ""

    for result in results:
        context += f"Source: {result['source']}\n"
        context += f"{result['text']}\n\n"

    return {
        "status": "success",
        "context": context,
        "sources": [result["source"] for result in results]
    }


# Test policy retrieval
policy_result = policy_agent(
    "SFO maximum surge multiplier and approval requirements"
)

print(policy_result["sources"])
print(policy_result["context"])

['sfo_pricing.md', 'lax_pricing.md', 'jfk_pricing.md']
Source: sfo_pricing.md
# SFO Pricing Policy

## Surge Pricing

The normal surge range is 1.0x to 1.5x.

The maximum permitted surge multiplier at SFO is 1.5x.

Surge increases above 1.3x require Operations Manager approval.

A surge multiplier above 1.5x is prohibited.

## Surge Conditions

Surge may be considered when request volume significantly exceeds available driver supply, completion rate falls below 85%, or airport queue conditions indicate insufficient supply.

Every surge adjustment must have a documented op

Source: lax_pricing.md
# LAX Pricing Policy

## Surge Pricing

The maximum permitted surge multiplier at LAX is 1.6x.

Surge above 1.3x requires Operations Manager approval.

Surge above 1.6x is prohibited.

## Surge Conditions
# LAX Pricing Policy

## Surge Pricing

The maximum permitted surge multiplier at LAX is 1.6x.

Surge above 1.3x requires Operations Manager approval.

Surge above 1.6x is prohibited.

## Surg

## 7. Resolution Agent

The Resolution Agent receives the operational investigation and applicable policy.

It evaluates the evidence and generates possible interventions.

The agent recommends an action but does not bypass policy or approval controls.
Sensitive execution and human approval will be implemented during Day 4.

In [11]:
# Generate a resolution recommendation from investigation and policy
def resolution_agent(investigation, policy_result):
    prompt = f"""
{resolution_instruction}

Operational Investigation:
{investigation}

Applicable Policy:
{policy_result["context"]}

Generate:
1. Issue
2. Evidence
3. Applicable Policy
4. Recommended Action
5. Reason
"""

    response = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=prompt
    )

    return response.text


# Generate a recommendation for the SFO investigation
recommendation = resolution_agent(
    investigation,
    policy_result
)

print(recommendation)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


**1. Issue**
Below-normal completion rate at SFO airport resulting in a medium severity operational issue.

**2. Evidence**
* Airport: SFO
* Completion Rate: 86% (0.86) *(Note: While the investigation summary lists 0.86, the SFO policy states surge may be considered when the completion rate falls below 85% [0.85]).*
* Average ETA: 12 minutes
* Active Drivers: 500
* Driver Cancellation Rate: 10% (0.1)
* Queue Size: 100
* Current Surge Multiplier: 1.1x

**3. Applicable Policy**
* **Source:** `sfo_pricing.md`
* **Surge Range & Limits:** The normal surge range is 1.0x to 1.5x. The maximum permitted surge at SFO is 1.5x. Surge increases above 1.3x require Operations Manager approval.
* **Surge Conditions:** Surge may be considered when request volume significantly exceeds available driver supply, completion rate falls below 85%, or airport queue conditions indicate insufficient supply. Every surge adjustment must have a documented operational reason.

**4. Recommended Action**
* **Intervent

## 8. Build the Orchestrator

The Orchestrator coordinates the specialized agents in the correct order.

It first sends the airport request to the Operations Investigator.
The investigation is then passed to the Policy & Compliance Agent.
Finally, the Resolution Agent uses both outputs to generate a recommendation.

This creates a controlled multi-agent workflow instead of one agent performing every responsibility.

In [12]:
# Coordinate the specialized agents for an airport investigation
def orchestrate_airport_issue(airport_code, proposed_action=None):
    # Step 1: investigate the airport using operational data
    investigation = investigate_airport(airport_code)

    if investigation["status"] == "error":
        return investigation

    # Step 2: retrieve policy related to the issue
    policy_query = f"""
    {airport_code} operational issue, completion rate, surge multiplier,
    maximum surge limit, and approval requirements
    """

    policy_result = policy_agent(policy_query)

    # Step 3: generate a resolution recommendation
    recommendation = resolution_agent(
        investigation,
        policy_result
    )

    return {
        "status": "success",
        "airport_code": airport_code,
        "investigation": investigation,
        "policy": policy_result,
        "recommendation": recommendation
    }


# Test the complete agent handoff
orchestration_result = orchestrate_airport_issue("SFO")

print("Airport:", orchestration_result["airport_code"])
print("\nInvestigation:")
print(orchestration_result["investigation"])

print("\nRecommendation:")
print(orchestration_result["recommendation"])

Airport: SFO

Investigation:
{'status': 'success', 'airport_code': 'SFO', 'metrics': {'status': 'success', 'airport_code': 'SFO', 'completion_rate': 0.86, 'average_eta': 12, 'active_drivers': 500, 'driver_cancellation_rate': 0.1, 'queue_size': 100, 'surge_multiplier': 1.1}, 'severity': 'medium', 'issue': 'Below-normal completion rate'}

Recommendation:
**1. Issue**
Below-normal completion rate at San Francisco International Airport (SFO).

**2. Evidence**
* **Airport Code:** SFO
* **Status/Severity:** Medium issue severity
* **Completion Rate:** 86% (Note: The raw metrics show 0.86, which is 86%. However, policy dictates that surge may be considered when the completion rate falls *below* 85%. Wait, let's look closer at the prompt: the issue stated is "Below-normal completion rate" with a completion rate of 86%, while the policy triggers surge when the completion rate falls below 85%. Regardless, the completion rate is experiencing strain given the medium severity status, a queue size o

## 9. Controlled ReAct-Style Loop

The ReAct pattern repeatedly selects an action, executes it, and observes the result.

In this project, the loop is controlled with a maximum of five iterations.
The trace records only the selected action and returned observation summary.

Detailed private reasoning is not stored or displayed.
The loop stops when the investigation is complete, a tool fails repeatedly, or the iteration limit is reached.

In [13]:
# Limit the number of agent actions to prevent uncontrolled loops
MAX_ITERATIONS = 5


def run_agent_loop(airport_code, proposed_action=None):
    # Store the visible workflow trace
    trace = []

    # Track the current investigation state
    state = {
        "airport_code": airport_code,
        "investigation": None,
        "policy": None,
        "recommendation": None
    }

    for iteration in range(1, MAX_ITERATIONS + 1):
        # Select the next action based on the current state
        if state["investigation"] is None:
            action = "investigate_airport"

            # Execute the operational investigation
            result = investigate_airport(airport_code)

            # Stop if the operational tool failed
            if result["status"] == "error":
                trace.append({
                    "iteration": iteration,
                    "action": action,
                    "observation": result["message"]
                })
                return {
                    "status": "error",
                    "trace": trace
                }

            state["investigation"] = result
            observation = "Airport investigation completed"

        elif state["policy"] is None:
            action = "retrieve_policy"

            # Retrieve policy based on the investigation
            policy_query = f"""
            {airport_code} completion rate, surge limits,
            approval requirements, and operational restrictions
            """

            result = policy_agent(policy_query)
            state["policy"] = result
            observation = "Relevant airport policy retrieved"

        elif state["recommendation"] is None:
            action = "generate_recommendation"

            # Generate a recommendation from the collected evidence
            state["recommendation"] = resolution_agent(
                state["investigation"],
                state["policy"]
            )

            observation = "Resolution recommendation generated"

        else:
            # Stop once all required stages are complete
            trace.append({
                "iteration": iteration,
                "action": "stop",
                "observation": "Investigation completed"
            })
            break

        # Record only the action and observation summary
        trace.append({
            "iteration": iteration,
            "action": action,
            "observation": observation
        })

    return {
        "status": "success",
        "state": state,
        "trace": trace
    }


# Run the controlled loop
loop_result = run_agent_loop("SFO")

for step in loop_result["trace"]:
    print(step)

{'iteration': 1, 'action': 'investigate_airport', 'observation': 'Airport investigation completed'}
{'iteration': 2, 'action': 'retrieve_policy', 'observation': 'Relevant airport policy retrieved'}
{'iteration': 3, 'action': 'generate_recommendation', 'observation': 'Resolution recommendation generated'}
{'iteration': 4, 'action': 'stop', 'observation': 'Investigation completed'}


## 10. Conversational Memory

Conversational memory allows the assistant to resolve references such as “its surge”
or “that airport” using the previous conversation.

The memory stores the latest airport mentioned and the latest investigation result.
Only relevant conversation state is retained for this demonstration.

This allows a follow-up question to use the correct airport without requiring the user to repeat its code.

In [14]:
# Store the latest conversation context
conversation_memory = {
    "last_airport": None,
    "last_investigation": None,
    "last_user_query": None
}


def update_memory(airport_code, user_query, investigation=None):
    # Save the latest context for follow-up questions
    conversation_memory["last_airport"] = airport_code
    conversation_memory["last_user_query"] = user_query
    conversation_memory["last_investigation"] = investigation


def resolve_airport_reference(user_query):
    # Detect an explicit airport code in the current question
    query = user_query.upper()

    for airport_code in ["SFO", "LAX", "JFK"]:
        if airport_code in query:
            return airport_code

    # Otherwise use the airport from the previous conversation
    return conversation_memory["last_airport"]


def ask_with_memory(user_query):
    # Resolve the airport from the current or previous question
    airport_code = resolve_airport_reference(user_query)

    if airport_code is None:
        return {
            "status": "error",
            "message": "Please specify an airport code."
        }

    # Run the agent workflow for the resolved airport
    result = run_agent_loop(airport_code)

    # Save the conversation context
    if result["status"] == "success":
        update_memory(
            airport_code,
            user_query,
            result["state"]["investigation"]
        )

    return result


# First user message
first_response = ask_with_memory("Check SFO")

print("First airport:", conversation_memory["last_airport"])

# Follow-up message uses the stored airport
second_response = ask_with_memory("What about its surge?")

print("Follow-up airport:", conversation_memory["last_airport"])
print(second_response["state"]["investigation"]["metrics"])

First airport: SFO
Follow-up airport: SFO
{'status': 'success', 'airport_code': 'SFO', 'completion_rate': 0.86, 'average_eta': 12, 'active_drivers': 500, 'driver_cancellation_rate': 0.1, 'queue_size': 100, 'surge_multiplier': 1.1}


## 11. End-to-End Agent Demonstration

The following example tests the complete Day 3 workflow.

The user asks the system to investigate an airport and consider a possible surge change.
The Orchestrator obtains operational information, retrieves applicable policy, and generates a recommendation.

The system does not directly bypass approval controls.
Actual permission checks and human approval will be added during Day 4.

In [15]:
# Run the complete Day 3 workflow for an airport
demo_result = run_agent_loop("SFO")

print("Workflow status:", demo_result["status"])

print("\nAgent trace:")
for step in demo_result["trace"]:
    print(step)

if demo_result["status"] == "success":
    print("\nInvestigation:")
    print(demo_result["state"]["investigation"])

    print("\nRecommendation:")
    print(demo_result["state"]["recommendation"])

Workflow status: success

Agent trace:
{'iteration': 1, 'action': 'investigate_airport', 'observation': 'Airport investigation completed'}
{'iteration': 2, 'action': 'retrieve_policy', 'observation': 'Relevant airport policy retrieved'}
{'iteration': 3, 'action': 'generate_recommendation', 'observation': 'Resolution recommendation generated'}
{'iteration': 4, 'action': 'stop', 'observation': 'Investigation completed'}

Investigation:
{'status': 'success', 'airport_code': 'SFO', 'metrics': {'status': 'success', 'airport_code': 'SFO', 'completion_rate': 0.86, 'average_eta': 12, 'active_drivers': 500, 'driver_cancellation_rate': 0.1, 'queue_size': 100, 'surge_multiplier': 1.1}, 'severity': 'medium', 'issue': 'Below-normal completion rate'}

Recommendation:
Based on the operational investigation findings and applicable policies, here is the resolution report:

### 1. Issue
Below-normal completion rate at SFO airport.

### 2. Evidence
* **Airport:** SFO
* **Completion Rate:** 86% (Note: Whi

## 12. Validate the Multi-Agent Workflow

The Day 3 system should correctly route an operational request through the appropriate agents.

These tests verify airport investigation, policy retrieval, resolution generation, and conversational memory.

The goal is to confirm that the agent workflow produces grounded outputs rather than simply generating an answer from the LLM.

In [16]:
# Test the main Day 3 workflow
test_airports = ["SFO", "LAX", "JFK"]

for airport in test_airports:
    result = run_agent_loop(airport)

    print(f"Airport: {airport}")
    print(f"Status: {result['status']}")

    if result["status"] == "success":
        print("Investigation:", result["state"]["investigation"]["issue"])
        print("Severity:", result["state"]["investigation"]["severity"])
        print("Recommendation generated:", bool(result["state"]["recommendation"]))

    print()

Airport: SFO
Status: success
Investigation: Below-normal completion rate
Severity: medium
Recommendation generated: True

Airport: LAX
Status: success
Investigation: Below-normal completion rate
Severity: medium
Recommendation generated: True

Airport: JFK
Status: success
Investigation: Below-normal completion rate
Severity: medium
Recommendation generated: True



## 13. Test Error Handling

Agentic systems must also handle invalid requests safely.

An invalid airport should not cause the workflow to continue with fabricated data.
The investigator should return the tool error and the orchestrator should stop the workflow.

This provides a basic failure boundary before the stronger guardrails introduced on Day 4.

In [17]:
# Test an invalid airport request
invalid_result = run_agent_loop("ABC")

print(invalid_result)

{'status': 'error', 'trace': [{'iteration': 1, 'action': 'investigate_airport', 'observation': 'Invalid airport code: ABC'}]}


## 14. Conversational Agent Demo

This example demonstrates how the assistant maintains context across multiple requests.

The first request identifies SFO as the airport.
The follow-up request refers to “its surge” without repeating the airport code.

The stored conversation state allows the system to resolve the reference back to SFO.

In [18]:
# Reset memory before the conversation demo
conversation_memory = {
    "last_airport": None,
    "last_investigation": None,
    "last_user_query": None
}

# User identifies the airport
response_1 = ask_with_memory("Check SFO")

print("User: Check SFO")
print("Resolved airport:", conversation_memory["last_airport"])

# User refers to the previous airport indirectly
response_2 = ask_with_memory("What about its surge?")

print("\nUser: What about its surge?")
print("Resolved airport:", conversation_memory["last_airport"])
print("Current surge:",
      response_2["state"]["investigation"]["metrics"]["surge_multiplier"])

User: Check SFO
Resolved airport: SFO

User: What about its surge?
Resolved airport: SFO
Current surge: 1.1


## 15. Day 3 Completion Summary

Day 3 converts the Day 1 RAG and Day 2 operational tools into a controlled agentic workflow.

The system contains an Operations Investigator, Policy & Compliance Agent, and Resolution Agent coordinated by an Orchestrator.

A controlled action-observation loop limits repeated execution, while conversational memory allows follow-up questions to retain airport context.

The resulting workflow can investigate an airport, retrieve relevant policy, and generate a grounded operational recommendation.

In [19]:
# Test that the new agent modules load correctly
from src.agents import (
    investigate_airport,
    policy_agent,
    resolution_agent,
    orchestrate_airport_issue,
    run_agent_loop
)

from src.memory import (
    conversation_memory,
    update_memory,
    resolve_airport_reference,
    clear_memory
)

print("Agent modules loaded successfully")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 755.27it/s]


Agent modules loaded successfully


## 16. Use the Packaged Agent Modules

The core agent logic has now been moved into the `src/` package.

The notebook will only prepare the policy knowledge base, Gemini client, and demonstration inputs.

This avoids maintaining separate copies of the same agent logic in the notebook and Python files.

The packaged modules can later be imported directly by the Flask application.

In [20]:
# Import the packaged agent workflow
from src.agents import (
    investigate_airport,
    policy_agent,
    resolution_agent,
    orchestrate_airport_issue,
    run_agent_loop
)

# Import the packaged conversation memory
from src.memory import (
    conversation_memory,
    update_memory,
    resolve_airport_reference,
    clear_memory
)

print("Packaged agent workflow loaded")

Packaged agent workflow loaded


## 17. Final Packaged Agent Test

The packaged workflow should produce the same result as the notebook prototype.

This test verifies that the agent modules can independently execute the investigation, policy retrieval, and resolution workflow.

The notebook is now acting as a demonstration layer rather than the implementation layer.

In [21]:
# Run the packaged agent workflow
final_test = run_agent_loop(
    "SFO",
    chunks,
    policy_index,
    client
)

print("Status:", final_test["status"])

for step in final_test["trace"]:
    print(step)

if final_test["status"] == "success":
    print("\nIssue:")
    print(final_test["state"]["investigation"]["issue"])

    print("\nRecommendation:")
    print(final_test["state"]["recommendation"])

Status: success
{'iteration': 1, 'action': 'investigate_airport', 'observation': 'Airport investigation completed'}
{'iteration': 2, 'action': 'retrieve_policy', 'observation': 'Relevant airport policy retrieved'}
{'iteration': 3, 'action': 'generate_recommendation', 'observation': 'Resolution recommendation generated'}
{'iteration': 4, 'action': 'stop', 'observation': 'Investigation completed'}

Issue:
Below-normal completion rate

Recommendation:
**1. Issue**
Below-normal completion rate at SFO airport (current completion rate is 86%, approaching/fluctuating near the operational threshold, with current metrics showing specific strain on supply and cancellations). *Note: While the investigation reports a completion rate of 0.86 (86%), the policy mandates an expected completion rate **above** 85% and explicitly notes that surge may be considered when the completion rate falls below 85% or when operational conditions indicate insufficient supply.*

**2. Evidence**
* Airport: SFO
* Sever

## 18. Test Packaged Conversational Memory

The memory module is tested separately from the agent logic.

The first request establishes the airport context.
The second request uses an indirect reference and resolves it using stored conversation state.

This confirms that conversational context is available to the packaged application components.

In [22]:
# Clear any previous conversation state
clear_memory()

# Store SFO as the current conversation context
update_memory(
    "SFO",
    "Check SFO"
)

print("Stored airport:", conversation_memory["last_airport"])

# Resolve an indirect follow-up reference
resolved_airport = resolve_airport_reference(
    "What about its surge?"
)

print("Resolved airport:", resolved_airport)

Stored airport: SFO
Resolved airport: SFO
